In [0]:
from pyspark.sql import functions as F
from datetime import datetime, timezone
from pyspark.sql.types import *

In [0]:
%run ./../config/00_project_config

In [0]:
%run ./../setup/00_storage_configuration

In [0]:
orders_watermark_name = "fact_orders_orders"
order_items_watermark_name = "fact_orders_order_items"
payments_watermark_name = "fact_orders_payments"
shipments_watermark_name = "fact_orders_shipments"

In [0]:
def get_current_watermark(df_watermarks, pipeline_name):
    watermark_record = df_watermarks \
    .filter(F.col("pipeline_name") == pipeline_name) \
    .first()

    watermark = (datetime(2000, 1, 1)) if watermark_record is None else watermark_record["last_processed_timestamp"]
    
    return watermark

In [0]:
df_watermarks = spark.read.format("delta") \
    .load(PIPELINE_WATERMARK_PATH) \
    .filter(F.col("pipeline_name").isin([orders_watermark_name, order_items_watermark_name, payments_watermark_name, shipments_watermark_name]))

In [0]:
orders_watermark = get_current_watermark(df_watermarks, orders_watermark_name)
order_items_watermark = get_current_watermark(df_watermarks, order_items_watermark_name)
payments_watermark = get_current_watermark(df_watermarks, payments_watermark_name)
shipments_watermark = get_current_watermark(df_watermarks, shipments_watermark_name)

In [0]:
df_orders_all = spark.read.format("delta") \
    .load(f"{SILVER_PATH}/orders")

In [0]:
df_updated_orders = df_orders_all \
    .filter(F.col("updated_at") > orders_watermark)

In [0]:
print(df_updated_orders.count())

In [0]:
df_order_items_all = spark.read.format("delta") \
    .load(f"{SILVER_PATH}/order_items")

In [0]:
df_updated_order_items = df_order_items_all \
    .filter(F.col("updated_at") > order_items_watermark)

In [0]:
df_customers = spark.read.format("delta") \
    .load(f"{SILVER_PATH}/customers")

In [0]:
df_payments_all = spark.read.format("delta") \
    .load(f"{SILVER_PATH}/payments")

In [0]:
df_updated_payments = df_payments_all \
    .filter(F.col("updated_at") > payments_watermark)

In [0]:
df_shipments_all = spark.read.format("delta") \
    .load(f"{SILVER_PATH}/shipments")

In [0]:
df_updated_shipments = df_shipments_all \
    .filter(F.col("updated_at") > shipments_watermark)

In [0]:
df_orders_to_process = df_updated_orders.select("order_id") \
    .union(df_updated_order_items.select("order_id")) \
    .union(df_updated_payments.select("order_id")) \
    .union(df_updated_shipments.select("order_id")) \
    .distinct()

In [0]:
print(df_orders_to_process.count())

In [0]:
df_orders_affected = df_orders_to_process.alias("otp").join(
    df_orders_all.alias("o"),
    "order_id"
).select(
    F.col("o.*")
)

In [0]:
df_agg_order_items = df_order_items_all.alias("oi") \
    .join(
        df_orders_to_process.alias("otp"),
        "order_id"
    ) \
    .groupBy("oi.order_id") \
    .agg(
        F.sum(F.col("quantity")).alias("total_quantity"),
        F.count("*").alias("order_item_count")
    )

In [0]:
%skip
display(df_agg_order_items)

In [0]:
df_payments_affected = df_orders_to_process.alias("op").join(
    df_payments_all.alias("p"),
    "order_id"
) \
    .select("p.*")

In [0]:
df_shipments_affected = df_orders_to_process.alias("op").join(
    df_shipments_all.alias("s"),
    "order_id"
) \
    .select("s.*")

In [0]:
df_orders_gold = df_orders_affected.alias("o") \
    .join(
        df_customers.alias("c"),
        (
            (F.col("c.customer_id") == F.col("o.customer_id")) &
            (F.col("o.order_date") >= F.col("c.valid_from")) &
            (
                (F.col("o.order_date") < F.col("c.valid_to")) |
                F.col("valid_to").isNull()
            )
        ),
    ) \
    .join(
        df_agg_order_items.alias("oi"),
        "order_id"
    ) \
    .join(
        df_payments_affected.alias("p"),
        "order_id"
    ) \
    .join(
        df_shipments_affected.alias("s"),
        "order_id",
        "left"
    ) \
    .select(
        F.col("o.order_id"),
        F.col("c.customer_sk"),
        F.col("o.order_date"),
        F.col("o.order_status"),
        F.col("o.total_amount"),
        F.col("oi.total_quantity"),
        F.col("oi.order_item_count"),
        F.col("p.amount").alias("payment_amount"),
        F.col("p.payment_method"),
        F.col("p.payment_status"),
        F.col("p.payment_date"),
        F.col("s.shipment_status"),
        F.col("s.shipment_date"),
        F.col("s.delivery_date"),
        F.col("s.warehouse_id"),
        F.col("s.tracking_number")
    )

In [0]:
%skip
display(df_orders_gold)

In [0]:
from delta.tables import DeltaTable

fact_orders_delta_table = DeltaTable.forPath(
    spark,
    f"{GOLD_PATH}/fact_orders"
)

fact_orders_delta_table.alias("target").merge(
    df_orders_gold.alias("source"),
    "target.order_id = source.order_id"
) \
    .whenMatchedUpdate(
        set = {
            'customer_sk': 'source.customer_sk',
            'order_date': 'source.order_date',
            'order_status': 'source.order_status',
            'total_amount': 'source.total_amount',
            'total_quantity': 'source.total_quantity',
            'order_item_count': 'source.order_item_count',
            'payment_amount': 'source.payment_amount',
            'payment_method': 'source.payment_method',
            'payment_status': 'source.payment_status',
            'payment_date': 'source.payment_date',
            'shipment_status': 'source.shipment_status',
            'shipment_date': 'source.shipment_date',
            'delivery_date': 'source.delivery_date',
            'warehouse_id': 'source.warehouse_id',
            'tracking_number': 'source.tracking_number'
        }
    ) \
    .whenNotMatchedInsertAll() \
    .execute()

In [0]:
%skip
df_orders_gold.write.format("delta").mode("append").save(f"{GOLD_PATH}/fact_orders")

In [0]:
def get_max_update_date_from_df(df, column_name):
    record = df \
        .agg(
            F.max(column_name).alias("updated_at")
        )
    return record.first()["updated_at"]

In [0]:
orders_last_processed_ts = orders_watermark if df_updated_orders.isEmpty() else get_max_update_date_from_df(df_updated_orders, "updated_at")
order_items_last_processed_ts = order_items_watermark if df_updated_order_items.isEmpty() else get_max_update_date_from_df(df_updated_order_items, "updated_at")
payments_last_processed_ts = payments_watermark if df_updated_payments.isEmpty() else get_max_update_date_from_df(df_updated_payments, "updated_at")
shipments_last_processed_ts = shipments_watermark if df_updated_shipments.isEmpty() else get_max_update_date_from_df(df_updated_shipments, "updated_at")

In [0]:
current_ts = datetime.now(timezone.utc)

watermark_records = [
    (orders_watermark_name, orders_last_processed_ts, current_ts, "SUCCESS", current_ts, current_ts),
    (order_items_watermark_name, order_items_last_processed_ts, current_ts, "SUCCESS", current_ts, current_ts),
    (payments_watermark_name, payments_last_processed_ts, current_ts, "SUCCESS", current_ts, current_ts),
    (shipments_watermark_name, shipments_last_processed_ts, current_ts, "SUCCESS", current_ts, current_ts)
]

df_updated_watermarks = spark.createDataFrame(
    watermark_records,
    [
        "pipeline_name",
        "last_processed_timestamp",
        "last_run_timestamp",
        "last_run_status",
        "created_at",
        "updated_at"
    ]
)

In [0]:
from delta.tables import DeltaTable

watermark_delta_table = DeltaTable.forPath(
    spark,
    PIPELINE_WATERMARK_PATH
)

watermark_delta_table.alias("target").merge(
    df_updated_watermarks.alias("source"),
    "target.pipeline_name = source.pipeline_name"
) \
    .whenMatchedUpdate(
        set = {
            "last_processed_timestamp": "source.last_processed_timestamp",
            "last_run_timestamp": "source.last_run_timestamp",
            "last_run_status": "source.last_run_status",
            "updated_at": "source.updated_at"
        }
    ) \
    .whenNotMatchedInsertAll() \
    .execute()

In [0]:
%skip
display(spark.read.format("delta").load(PIPELINE_WATERMARK_PATH))